# CMEMS Sea Level Data — Bay of Bengal Analysis & Visualization 2024

This notebook provides a modular, memory-efficient workflow for exploring
CMEMS global sea-level (L4, 0.125°) daily data over the **Bay of Bengal**
for the year **2024**.

Key features:
- Lazy-loaded, chunked `xarray` datasets via `open_mfdataset`
- Generic bounding-box subsetting
- Daily maps, monthly-mean panels, area-averaged time series, and anomaly maps
- Minimal cartopy usage — lightweight `imshow` plots with geographic extents

## 1. Import Libraries & Configuration

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Optional: coastlines overlay
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False

# ── Configuration ────────────────────────────────────────────────────────────────
ROOT_DIR = (
    r"C:\Users\abhik\Downloads\"
    r"SEALEVEL_GLO_PHY_L4_MY_008_047 "
    r"cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D 2024 01"
)
YEAR = 2024

# Bay of Bengal bounding box
BBOX_BOB = dict(lat_min=5, lat_max=25, lon_min=80, lon_max=100)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150})

## 2. Data Loading — `open_year_dataset()`

In [ ]:
def open_year_dataset(root_dir, year,
                      chunks={"time": 10, "latitude": 360, "longitude": 720}):
    """Lazily open every NetCDF file under *root_dir*/01 … 12."""
    month_dirs = [f"{m:02d}" for m in range(1, 13)]
    file_list = []
    for m in month_dirs:
        pattern = os.path.join(root_dir, m, "*.nc")
        files = sorted(glob.glob(pattern))
        file_list.extend(files)
    if not file_list:
        raise FileNotFoundError(
            f"No .nc files found under {root_dir}/01..12. "
            f"Check ROOT_DIR and folder structure."
        )
    print(f"Found {len(file_list)} files across 12 months.")
    ds = xr.open_mfdataset(
        file_list,
        combine="nested",
        concat_dim="time",
        chunks=chunks,
        engine="netcdf4",
    )
    ds = ds.assign_coords(month=("time", ds["time"].dt.month.values))
    return ds

## 3. Spatial Subsetting — `subset_bbox()`

In [ ]:
def subset_bbox(ds, lat_min, lat_max, lon_min, lon_max):
    """Return the dataset clipped to the given bounding box."""
    return ds.sel(
        latitude=slice(lat_min, lat_max),
        longitude=slice(lon_min, lon_max),
    )

## 4. Load Dataset & Subset to Bay of Bengal

In [ ]:
ds_global = open_year_dataset(ROOT_DIR, YEAR)
ds = subset_bbox(ds_global, **BBOX_BOB)

print("─" * 60)
print(f"Bay of Bengal subset  lat {BBOX_BOB['lat_min']}–{BBOX_BOB['lat_max']}°N, "
      f"lon {BBOX_BOB['lon_min']}–{BBOX_BOB['lon_max']}°E")
print(f"Time steps : {ds.sizes['time']}")
print(f"Grid       : {ds.sizes['latitude']} × {ds.sizes['longitude']}")
print(f"Variables  : {list(ds.data_vars)}")
print("─" * 60)
ds

## 5. Daily Maps for a Selected Month

In [ ]:
def plot_daily_month(ds, year, month, var_name="sla",
                     ncols=5, figsize=(18, 14), cmap="RdBu_r"):
    """Plot each day of *month* as a small-multiple panel using imshow."""
    dsm = ds.sel(time=ds["time"].dt.month == month)
    da = dsm[var_name]
    nt = da.sizes["time"]
    nrows = int(np.ceil(nt / ncols))

    vmin = float(da.quantile(0.02))
    vmax = float(da.quantile(0.98))
    extent = [
        float(da.longitude.min()), float(da.longitude.max()),
        float(da.latitude.min()),  float(da.latitude.max()),
    ]

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).flatten()

    for i in range(nt):
        ax = axes[i]
        day_data = da.isel(time=i).values
        date_str = pd.Timestamp(da.time.values[i]).strftime("%d %b")
        im = ax.imshow(
            day_data, origin="lower", extent=extent,
            cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto",
        )
        ax.set_title(date_str, fontsize=8)
        ax.tick_params(labelsize=6)

    for j in range(nt, len(axes)):
        fig.delaxes(axes[j])

    fig.colorbar(
        im, ax=axes[:nt].tolist(), orientation="horizontal",
        shrink=0.6, label=f"{var_name.upper()} (m)", pad=0.04,
    )
    fig.suptitle(
        f"{var_name.upper()} Daily — {year}-{month:02d} — Bay of Bengal",
        fontsize=14,
    )
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    return fig


# Example: January
fig = plot_daily_month(ds, YEAR, month=1)
plt.show()

## 6. Monthly Mean Maps (3×4 Panel)

In [ ]:
def plot_monthly_means(ds, year, var_name="sla",
                       nrows=3, ncols=4, figsize=(16, 10), cmap="RdBu_r"):
    """3×4 panel of monthly-mean maps using imshow."""
    monthly = ds[var_name].groupby("time.month").mean(dim="time")
    vmin = float(monthly.quantile(0.02))
    vmax = float(monthly.quantile(0.98))
    extent = [
        float(ds.longitude.min()), float(ds.longitude.max()),
        float(ds.latitude.min()),  float(ds.latitude.max()),
    ]
    month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten()

    for i in range(12):
        ax = axes[i]
        data = monthly.sel(month=i + 1).values
        im = ax.imshow(
            data, origin="lower", extent=extent,
            cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto",
        )
        ax.set_title(month_names[i], fontsize=10)
        ax.tick_params(labelsize=6)

    fig.colorbar(
        im, ax=axes.tolist(), orientation="horizontal",
        shrink=0.6, label=f"{var_name.upper()} (m)", pad=0.04,
    )
    fig.suptitle(
        f"Monthly Mean {var_name.upper()} — Bay of Bengal {year}", fontsize=14,
    )
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    return fig


fig = plot_monthly_means(ds, YEAR)
plt.show()

## 7. Area-Averaged Time Series

In [ ]:
def plot_area_timeseries(ds, var_name="sla", figsize=(14, 5)):
    """Daily area-averaged time series, weighted by cos(latitude)."""
    weights = np.cos(np.deg2rad(ds.latitude))
    da_weighted = ds[var_name].weighted(weights)
    ts = da_weighted.mean(dim=["latitude", "longitude"])
    ts_pd = ts.to_series()

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(ts_pd.index, ts_pd.values, lw=0.8, color="steelblue")
    annual_mean = ts_pd.mean()
    ax.axhline(annual_mean, ls="--", color="red", lw=0.7,
               label=f"Annual mean = {annual_mean:.4f} m")
    ax.fill_between(ts_pd.index, ts_pd.values, annual_mean, alpha=0.15)
    ax.set_xlabel("Date")
    ax.set_ylabel(f"{var_name.upper()} (m)")
    ax.set_title(
        f"Area-Averaged {var_name.upper()} — Bay of Bengal "
        f"{int(ds.time.dt.year.values[0])}"
    )
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig


fig = plot_area_timeseries(ds)
plt.show()

## 8. Anomaly Maps (Relative to Annual Mean)

In [ ]:
def plot_monthly_anomalies(ds, year, var_name="sla",
                           nrows=3, ncols=4, figsize=(16, 10),
                           cmap="RdBu_r"):
    """Monthly anomaly maps relative to the annual mean."""
    annual_mean = ds[var_name].mean(dim="time")
    monthly = ds[var_name].groupby("time.month").mean(dim="time")
    anomalies = monthly - annual_mean

    vabs = float(np.abs(anomalies).quantile(0.98))
    extent = [
        float(ds.longitude.min()), float(ds.longitude.max()),
        float(ds.latitude.min()),  float(ds.latitude.max()),
    ]
    month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten()

    for i in range(12):
        ax = axes[i]
        anom = anomalies.sel(month=i + 1).values
        im = ax.imshow(
            anom, origin="lower", extent=extent,
            cmap=cmap, vmin=-vabs, vmax=vabs, aspect="auto",
        )
        ax.set_title(month_names[i], fontsize=10)
        ax.tick_params(labelsize=6)

    fig.colorbar(
        im, ax=axes.tolist(), orientation="horizontal",
        shrink=0.6, label=f"{var_name.upper()} anomaly (m)",
    )
    fig.suptitle(
        f"{var_name.upper()} Monthly Anomaly (vs Annual Mean) — "
        f"Bay of Bengal {year}",
        fontsize=14,
    )
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    return fig


fig = plot_monthly_anomalies(ds, YEAR)
plt.show()

## 9. Custom Region Example

In [ ]:
# Central Bay of Bengal sub-region
BBOX_CUSTOM = dict(lat_min=10, lat_max=18, lon_min=85, lon_max=95)
ds_custom = subset_bbox(ds_global, **BBOX_CUSTOM)

fig = plot_monthly_means(ds_custom, YEAR, var_name="adt")
plt.show()

fig2 = plot_area_timeseries(ds_custom, var_name="sla")
plt.show()